# LeetCode 675 — Cut Off Trees for Golf Event bằng Python

- Link: https://leetcode.com/problems/cut-off-trees-for-golf-event/
- Độ khó: **Hard**
- Chủ đề: BFS trên lưới, Đường đi ngắn nhất, A* (A-star), 0-1 BFS (thuật toán Hadlock), Sắp xếp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def cutOffTree(self, forest: List[List[int]]) -> int:

```

## 1. Giới thiệu & mục tiêu học

Bài này nghe có vẻ phức tạp (chặt cây theo thứ tự, lưới có vật cản), nhưng sau khi "lột vỏ" nó chỉ là: **sắp xếp các cây theo chiều cao, rồi cộng các khoảng cách ngắn nhất giữa hai cây liên tiếp**. Thử thách thật sự là cài đặt **tìm đường ngắn nhất trên lưới** cho đúng và đủ nhanh. Đây là dịp tốt để học BFS thật chắc, rồi nâng cấp lên A* và 0-1 BFS.

Sau notebook này bạn sẽ:

1. Nhận ra vì sao thứ tự chặt là **cố định** và vì sao chặt cây **không làm thay đổi** đường đi (cây vẫn đi qua được).
2. Viết BFS trên lưới chuẩn: hàng đợi, mảng `visited`, 4 hướng, kiểm tra biên.
3. Dùng Floyd–Warshall làm brute force đối chiếu cho lưới nhỏ.
4. Hiểu và cài đặt **A*** với heuristic Manhattan, và **thuật toán Hadlock** (0-1 BFS đếm số bước "đi vòng").
5. Kiểm chứng tất cả trên lưới ngẫu nhiên có seed và biết các edge case (không tới được, ô xuất phát là cây, ...).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho lưới `forest` kích thước `m × n`:

- `0`: vật cản, **không** đi được.
- `1`: ô trống, đi được.
- Số `> 1`: một **cây** có chiều cao bằng số đó; ô này **đi được**.

Mỗi bước bạn đi sang một ô kề (trên, dưới, trái, phải). Đứng ở ô có cây thì có thể chặt; sau khi chặt, ô thành `1`. Bạn bắt đầu ở `(0, 0)` và phải chặt **mọi** cây theo thứ tự **chiều cao tăng dần**. Trả về **tổng số bước ít nhất**, hoặc `-1` nếu không thể chặt hết.

Đảm bảo: không có hai cây cao bằng nhau, và có ít nhất một cây cần chặt.

**Ví dụ 1**

```text
Input:  forest = [[1,2,3],
                  [0,0,4],
                  [7,6,5]]
Output: 6
Giải thích: đi (0,0)->(0,1) chặt 2, ->(0,2) chặt 3, ->(1,2) chặt 4, ->(2,2) chặt 5,
            ->(2,1) chặt 6, ->(2,0) chặt 7. Tổng 6 bước.
```

**Ví dụ 2**

```text
Input:  forest = [[1,2,3],
                  [0,0,0],
                  [7,6,5]]
Output: -1
Giải thích: hàng giữa bị chặn hoàn toàn, không tới được các cây ở hàng dưới.
```

**Ví dụ 3**

```text
Input:  forest = [[2,3,4],
                  [0,0,5],
                  [8,7,6]]
Output: 6
Giải thích: đứng sẵn ở cây 2 — chặt ngay, không tốn bước. Sau đó đi như ví dụ 1.
```

**Ràng buộc**

- `m == forest.length`, `n == forest[i].length`, `1 <= m, n <= 50`
- `0 <= forest[i][j] <= 10^9`
- Chiều cao các cây đôi một khác nhau.

In [1]:
ex1 = [[1, 2, 3], [0, 0, 4], [7, 6, 5]]
ex2 = [[1, 2, 3], [0, 0, 0], [7, 6, 5]]
ex3 = [[2, 3, 4], [0, 0, 5], [8, 7, 6]]

def show(forest):
    for row in forest:
        print(' '.join('##' if v == 0 else (' .' if v == 1 else f"{v:>2}") for v in row))

for name, f in [("ex1", ex1), ("ex2", ex2), ("ex3", ex3)]:
    print(name)
    show(f)

ex1
 .  2  3
## ##  4
 7  6  5
ex2
 .  2  3
## ## ##
 7  6  5
ex3
 2  3  4
## ##  5
 8  7  6


Quy ước hiển thị: `##` là vật cản, `.` là ô trống, số là chiều cao cây.

## 3. Kiến thức nền

### 3.1. Duyệt 4 hướng trên lưới và kiểm tra biên

Từ ô `(r, c)`, bốn ô kề là `(r±1, c)` và `(r, c±1)`. Trước khi dùng phải kiểm tra **nằm trong lưới** và **không phải vật cản**.

In [2]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def neighbors(forest, r, c):
    m, n = len(forest), len(forest[0])
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n and forest[nr][nc] != 0:
            yield nr, nc

print("kề của (0,0) trong ex1:", list(neighbors(ex1, 0, 0)))
print("kề của (0,2) trong ex1:", list(neighbors(ex1, 0, 2)))

kề của (0,0) trong ex1: [(0, 1)]
kề của (0,2) trong ex1: [(1, 2), (0, 1)]


### 3.2. BFS với `deque` — đường ngắn nhất khi mỗi bước tốn 1

BFS lấy ô từ **đầu** hàng đợi, thêm ô mới vào **cuối**. Các ô được lấy ra theo khoảng cách tăng dần (0, 1, 1, 2, 2, 2, ...), nên lần đầu tới đích là đường ngắn nhất. Phải đánh dấu `visited` **khi thêm vào hàng đợi** để không thêm một ô nhiều lần.

In [3]:
from collections import deque

def bfs_dist(forest, sr, sc, tr, tc):
    if (sr, sc) == (tr, tc):
        return 0
    m, n = len(forest), len(forest[0])
    seen = [[False] * n for _ in range(m)]
    seen[sr][sc] = True
    q = deque([(sr, sc, 0)])
    while q:
        r, c, d = q.popleft()
        for nr, nc in neighbors(forest, r, c):
            if not seen[nr][nc]:
                if (nr, nc) == (tr, tc):
                    return d + 1
                seen[nr][nc] = True
                q.append((nr, nc, d + 1))
    return -1

print("ex1: (0,0) -> (2,0):", bfs_dist(ex1, 0, 0, 2, 0))
print("ex2: (0,0) -> (2,0):", bfs_dist(ex2, 0, 0, 2, 0))

ex1: (0,0) -> (2,0): 6
ex2: (0,0) -> (2,0): -1


### 3.3. Sắp xếp các cây theo chiều cao

Gom `(chiều cao, hàng, cột)` của mọi ô `> 1` rồi `sorted` — tuple so sánh theo phần tử đầu tiên là chiều cao.

In [4]:
trees = sorted((v, r, c) for r, row in enumerate(ex1) for c, v in enumerate(row) if v > 1)
print(trees)

[(2, 0, 1), (3, 0, 2), (4, 1, 2), (5, 2, 2), (6, 2, 1), (7, 2, 0)]


### 3.4. `heapq` cho A* và `deque.appendleft` cho 0-1 BFS

- **A*** cần lấy ra ô có `f = g + h` nhỏ nhất → min-heap.
- **0-1 BFS** (cạnh trọng số 0 hoặc 1): cạnh 0 thêm vào **đầu** deque (`appendleft`), cạnh 1 thêm vào **cuối** (`append`).

In [5]:
import heapq
h = []
for item in [(5, 'B'), (3, 'A'), (7, 'C')]:
    heapq.heappush(h, item)
print("heap lấy ra:", [heapq.heappop(h) for _ in range(3)])

dq = deque()
dq.append((1, 'cạnh 1'))
dq.appendleft((0, 'cạnh 0'))
print("deque:", list(dq))

heap lấy ra: [(3, 'A'), (5, 'B'), (7, 'C')]
deque: [(0, 'cạnh 0'), (1, 'cạnh 1')]


### 3.5. Khoảng cách Manhattan

`|r1 - r2| + |c1 - c2|` — số bước ít nhất nếu **không có vật cản**. Đây là **cận dưới** của khoảng cách thật, dùng làm heuristic cho A* và làm "phần cố định" trong Hadlock.

In [6]:
def manhattan(r1, c1, r2, c2):
    return abs(r1 - r2) + abs(c1 - c2)

print("Manhattan (0,0)-(2,0):", manhattan(0, 0, 2, 0), "| BFS thật trong ex1:", bfs_dist(ex1, 0, 0, 2, 0))

Manhattan (0,0)-(2,0): 2 | BFS thật trong ex1: 6


Ví dụ trên: Manhattan nói 2 bước nhưng thực tế phải đi vòng qua hàng trên → 6 bước. Chênh lệch `6 - 2 = 4 = 2 × 2` — tức đi vòng **2 lần** (mỗi bước đi "xa đích" phải bù lại bằng một bước "về gần"). Đây chính là ý tưởng của Hadlock.

## 4. Ý tưởng nền tảng / trực giác

### 4.1. Thứ tự chặt là cố định

Đề bắt chặt theo chiều cao tăng dần, chiều cao đôi một khác nhau → chỉ có **một** thứ tự: `T1, T2, ..., Tk`. Ta không phải chọn thứ tự (khác hẳn bài người du lịch TSP).

### 4.2. Chặt cây không làm thay đổi đường đi

Cây là ô **đi được**; sau khi chặt thành `1`, vẫn đi được. Tập ô đi được **không đổi** suốt quá trình. Vậy khoảng cách ngắn nhất giữa hai ô là **cố định**, và:

```text
đáp án = dist(start, T1) + dist(T1, T2) + ... + dist(T(k-1), Tk)
```

Nếu một đoạn nào đó không tới được → `-1`.

### 4.3. Tính tay ví dụ 1

Thứ tự: `2 (0,1)`, `3 (0,2)`, `4 (1,2)`, `5 (2,2)`, `6 (2,1)`, `7 (2,0)`.

| Đoạn | Từ | Tới | dist |
|---|---|---|---|
| start → 2 | (0,0) | (0,1) | 1 |
| 2 → 3 | (0,1) | (0,2) | 1 |
| 3 → 4 | (0,2) | (1,2) | 1 |
| 4 → 5 | (1,2) | (2,2) | 1 |
| 5 → 6 | (2,2) | (2,1) | 1 |
| 6 → 7 | (2,1) | (2,0) | 1 |
| **Tổng** | | | **6** |

### 4.4. Một ví dụ mà thứ tự bắt đi vòng

`forest = [[1, 4, 1], [3, 0, 2]]`: thứ tự `2 (1,2)`, `3 (1,0)`, `4 (0,1)`. Từ `(0,0)` tới `(1,2)`: 3 bước; từ `(1,2)` về `(1,0)`: phải vòng qua hàng trên, 4 bước; từ `(1,0)` tới `(0,1)`: 2 bước. Tổng 9 — dù đi một vòng tham lam theo vị trí thì ngắn hơn nhiều, nhưng đề bắt theo chiều cao.

In [7]:
def total_steps_with(dist_fn, forest):
    trees = sorted((v, r, c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1)
    sr, sc, total = 0, 0, 0
    for v, tr, tc in trees:
        d = dist_fn(forest, sr, sc, tr, tc)
        print(f"  ({sr},{sc}) -> cây {v} tại ({tr},{tc}): {d} bước")
        if d < 0:
            return -1
        total += d
        sr, sc = tr, tc
    return total

ex4 = [[1, 4, 1], [3, 0, 2]]
show(ex4)
print("Tổng:", total_steps_with(bfs_dist, ex4))

 .  4  .
 3 ##  2
  (0,0) -> cây 2 tại (1,2): 3 bước
  (1,2) -> cây 3 tại (1,0): 4 bước
  (1,0) -> cây 4 tại (0,1): 2 bước
Tổng: 9


## 5. Cách 1 — Brute Force: Floyd–Warshall trên mọi ô

**Ý tưởng.** Coi mỗi ô đi được là một đỉnh, hai ô kề là cạnh độ dài 1. Tính **khoảng cách giữa mọi cặp đỉnh** bằng Floyd–Warshall (ba vòng lặp lồng nhau), rồi cộng theo thứ tự cây.

Floyd–Warshall rất ngắn và hầu như không thể sai → dùng làm "trọng tài" cho lưới nhỏ.

```text
dist[u][v] = 0 nếu u == v, 1 nếu kề, vô cùng nếu không
với mỗi w: với mỗi u: với mỗi v: dist[u][v] = min(dist[u][v], dist[u][w] + dist[w][v])
```

In [8]:
def cut_off_floyd(forest):
    m, n = len(forest), len(forest[0])
    cells = [(r, c) for r in range(m) for c in range(n) if forest[r][c] != 0]
    if forest[0][0] == 0:
        cells.append((0, 0))               # xuất phát trên vật cản: vẫn là một đỉnh (cô lập)
    idx = {cell: i for i, cell in enumerate(cells)}
    V = len(cells)
    INF = float('inf')
    dist = [[INF] * V for _ in range(V)]
    for (r, c), i in idx.items():
        dist[i][i] = 0
        if forest[r][c] == 0:
            continue
        for nr, nc in neighbors(forest, r, c):
            dist[i][idx[(nr, nc)]] = 1
    for w in range(V):
        dw = dist[w]
        for u in range(V):
            duw = dist[u][w]
            if duw == INF:
                continue
            du = dist[u]
            for v in range(V):
                if duw + dw[v] < du[v]:
                    du[v] = duw + dw[v]
    trees = sorted((v, r, c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1)
    cur, total = idx[(0, 0)], 0
    for v, r, c in trees:
        d = dist[cur][idx[(r, c)]]
        if d == INF:
            return -1
        total += d
        cur = idx[(r, c)]
    return total

print(cut_off_floyd(ex1), cut_off_floyd(ex2), cut_off_floyd(ex3), cut_off_floyd(ex4))
assert cut_off_floyd(ex1) == 6 and cut_off_floyd(ex2) == -1 and cut_off_floyd(ex3) == 6

6 -1 6 9


**Độ phức tạp.** `V <= m·n = 2500` đỉnh → `O(V^3) = 1,5·10^10` phép — quá chậm. Space `O(V^2)`.

**Nút thắt.** Ta tính khoảng cách giữa **mọi cặp** ô, nhưng chỉ cần `k` cặp (các cây liên tiếp). Mỗi cặp tính riêng bằng BFS chỉ tốn `O(m·n)`.

In [9]:
import time, random
random.seed(0)
def rand_forest(m, n, p_block=0.2, p_tree=0.3):
    heights = random.sample(range(2, 10 * m * n + 10), m * n)
    g = []
    for r in range(m):
        row = []
        for c in range(n):
            x = random.random()
            row.append(0 if x < p_block else (heights[r * n + c] if x < p_block + p_tree else 1))
        g.append(row)
    if all(v <= 1 for row in g for v in row):
        g[m - 1][n - 1] = heights[-1]
    return g

for size in (6, 9, 12):
    g = rand_forest(size, size)
    t = time.perf_counter()
    cut_off_floyd(g)
    print(f"{size}x{size}: {time.perf_counter() - t:.3f}s")

6x6: 0.001s
9x9: 0.005s
12x12: 0.035s


## 6. Bước đột phá

**Insight 1 (tách bài toán).** Theo 4.1 và 4.2: đáp án = tổng `k` khoảng cách ngắn nhất độc lập. Bài toán Hard thu về: "**tính nhanh khoảng cách ngắn nhất giữa hai ô trên lưới có vật cản**", lặp `k` lần.

**Insight 2 (BFS đủ tốt).** Mỗi BFS tốn `O(m·n) = 2500`; `k <= 2500` cây → `6,25·10^6` lượt thăm ô. Chấp nhận được.

**Insight 3 (tận dụng hình học).** Lưới có cấu trúc: Manhattan là cận dưới. A* dùng nó để **ưu tiên hướng về đích**; Hadlock nhận xét `dist = Manhattan + 2 × (số bước đi xa đích)` và chỉ cần **tối thiểu số bước đi xa** bằng 0-1 BFS.

**Invariant khi cộng dồn:** sau khi xử lý `i` cây đầu, `total` = số bước ít nhất để chặt xong `i` cây đầu và đứng tại cây thứ `i`.

**Quy tắc nếu…thì…:**
- Nếu một BFS không tới được cây kế tiếp thì trả `-1` ngay.
- Nếu đứng sẵn ở cây (ô xuất phát là cây thấp nhất) thì khoảng cách 0.
- Nếu `forest[0][0] == 0` (xuất phát trên vật cản) thì không đi đâu được → `-1` (trừ khi cây duy nhất... không thể, vì ô 0 không phải cây).

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Time | Space |
|---|---|---|---|
| 1 | Floyd–Warshall (mục 5) | `O((mn)^3)` | `O((mn)^2)` |
| 2 | BFS cho từng cặp cây | `O(k · mn)` | `O(mn)` |
| 3 | A* (heuristic Manhattan) cho từng cặp | `O(k · mn log(mn))` xấu nhất, thường nhanh hơn | `O(mn)` |
| 4 | Hadlock (0-1 BFS theo số bước đi vòng) | `O(k · mn)` | `O(mn)` |

### 7.1. Cách 2 — BFS cho từng cặp cây

**Giải thích.** Sắp các cây; với mỗi cây kế tiếp chạy `bfs_dist` từ vị trí hiện tại. Cộng dồn, gặp `-1` thì dừng.

In [10]:
def cut_off_bfs(forest):
    trees = sorted((v, r, c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1)
    sr = sc = total = 0
    for _, tr, tc in trees:
        d = bfs_dist(forest, sr, sc, tr, tc)
        if d < 0:
            return -1
        total += d
        sr, sc = tr, tc
    return total

print(cut_off_bfs(ex1), cut_off_bfs(ex2), cut_off_bfs(ex3), cut_off_bfs(ex4))
assert cut_off_bfs(ex1) == 6 and cut_off_bfs(ex2) == -1

6 -1 6 9


**Đọc code `bfs_dist` (mục 3.2) từng dòng:**

- `if (sr, sc) == (tr, tc): return 0` — xử lý trường hợp đứng sẵn ở cây (ví dụ 3).
- `seen[sr][sc] = True` trước khi đưa vào hàng đợi.
- Kiểm tra đích **ngay khi phát hiện** ô kề (trả `d + 1`) — tiết kiệm một lớp BFS.
- Hết hàng đợi mà chưa gặp đích → `-1`.

**Lưu ý `forest[0][0] == 0`:** `bfs_dist` vẫn mở rộng từ `(0,0)` sang ô kề đi được — nghĩa là "nhảy ra" khỏi vật cản. Đề LeetCode không có test như vậy, nhưng để chặt chẽ, bản nộp kiểm tra riêng.

**Code nộp LeetCode (Cách 2):** dùng mảng 1 chiều và "dấu thời gian" để khỏi tạo lại `seen` mỗi lần BFS.

In [11]:
from typing import List

class SolutionBFS:
    def cutOffTree(self, forest: List[List[int]]) -> int:
        m, n = len(forest), len(forest[0])
        if forest[0][0] == 0:
            return -1
        trees = sorted((v, r * n + c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1)
        walk = [forest[i // n][i % n] != 0 for i in range(m * n)]
        stamp = [0] * (m * n)                # stamp[i] == lần BFS hiện tại => đã thăm
        cur_stamp = 0

        def bfs(src, dst):
            nonlocal cur_stamp
            if src == dst:
                return 0
            cur_stamp += 1
            stamp[src] = cur_stamp
            frontier, d = [src], 0
            while frontier:
                d += 1
                nxt = []
                for u in frontier:
                    r, c = divmod(u, n)
                    for v, ok in ((u - n, r > 0), (u + n, r < m - 1), (u - 1, c > 0), (u + 1, c < n - 1)):
                        if ok and walk[v] and stamp[v] != cur_stamp:
                            if v == dst:
                                return d
                            stamp[v] = cur_stamp
                            nxt.append(v)
                frontier = nxt
            return -1

        total, cur = 0, 0
        for _, pos in trees:
            d = bfs(cur, pos)
            if d < 0:
                return -1
            total += d
            cur = pos
        return total

assert SolutionBFS().cutOffTree(ex1) == 6
assert SolutionBFS().cutOffTree(ex2) == -1
assert SolutionBFS().cutOffTree(ex3) == 6
print("SolutionBFS OK")

SolutionBFS OK


**Hai mẹo tăng tốc trong bản nộp:**

- **Đánh số ô `r · n + c`**: một số nguyên thay cho tuple, `divmod` để lấy lại hàng/cột.
- **Dấu thời gian (`stamp`)**: thay vì tạo mảng `seen` mới mỗi BFS (`O(mn)` cấp phát), ta tăng `cur_stamp` và coi ô "đã thăm" khi `stamp[v] == cur_stamp`.
- **BFS theo lớp** (`frontier` → `nxt`): khoảng cách `d` là số lớp, không cần lưu trong hàng đợi.

**Độ phức tạp Cách 2:** Time `O(k · m · n)` (+ `O(mn log mn)` sắp xếp), Space `O(m · n)`.

### 7.2. Cách 3 — A* với heuristic Manhattan

**Giải thích.** A* giống Dijkstra nhưng ưu tiên ô có `f = g + h` nhỏ nhất, với `g` = số bước đã đi, `h` = Manhattan tới đích. Vì `h` là cận dưới **và** thỏa bất đẳng thức tam giác (đi một bước thì `h` đổi tối đa 1 — gọi là heuristic *nhất quán*), lần đầu lấy đích ra khỏi heap là đường ngắn nhất. Trên lưới thoáng, A* gần như đi thẳng tới đích, thăm rất ít ô.

In [12]:
def astar_dist(forest, sr, sc, tr, tc):
    if (sr, sc) == (tr, tc):
        return 0
    best_g = {(sr, sc): 0}
    heap = [(manhattan(sr, sc, tr, tc), 0, sr, sc)]
    while heap:
        f, g, r, c = heapq.heappop(heap)
        if (r, c) == (tr, tc):
            return g
        if g > best_g[(r, c)]:
            continue                                # bản ghi cũ, đã có đường tốt hơn
        for nr, nc in neighbors(forest, r, c):
            ng = g + 1
            if ng < best_g.get((nr, nc), float('inf')):
                best_g[(nr, nc)] = ng
                heapq.heappush(heap, (ng + manhattan(nr, nc, tr, tc), ng, nr, nc))
    return -1

def cut_off_astar(forest):
    return total_steps_quiet(astar_dist, forest)

def total_steps_quiet(dist_fn, forest):
    trees = sorted((v, r, c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1)
    sr = sc = total = 0
    for _, tr, tc in trees:
        d = dist_fn(forest, sr, sc, tr, tc)
        if d < 0:
            return -1
        total += d
        sr, sc = tr, tc
    return total

print(cut_off_astar(ex1), cut_off_astar(ex2), cut_off_astar(ex3), cut_off_astar(ex4))
assert cut_off_astar(ex1) == 6 and cut_off_astar(ex2) == -1

6 -1 6 9


**Đọc code:**

- Heap chứa `(f, g, r, c)`: so sánh theo `f` trước, hòa thì `g` nhỏ trước.
- `best_g` lưu `g` tốt nhất đã biết cho mỗi ô. Một ô có thể được đẩy vào heap nhiều lần; bản ghi có `g` lớn hơn `best_g` là "cũ" → bỏ qua (**lazy deletion**).
- Kiểm tra đích **khi lấy ra** (không phải khi đẩy vào) — với A*, chỉ lúc lấy ra mới chắc chắn tối ưu.

**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionAStar:
    def cutOffTree(self, forest: List[List[int]]) -> int:
        m, n = len(forest), len(forest[0])
        if forest[0][0] == 0:
            return -1
        trees = sorted((v, r, c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1)

        def astar(sr, sc, tr, tc):
            if (sr, sc) == (tr, tc):
                return 0
            best = {(sr, sc): 0}
            heap = [(abs(sr - tr) + abs(sc - tc), 0, sr, sc)]
            while heap:
                _, g, r, c = heapq.heappop(heap)
                if (r, c) == (tr, tc):
                    return g
                if g > best[(r, c)]:
                    continue
                for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                    if 0 <= nr < m and 0 <= nc < n and forest[nr][nc]:
                        ng = g + 1
                        if ng < best.get((nr, nc), 1 << 30):
                            best[(nr, nc)] = ng
                            heapq.heappush(heap, (ng + abs(nr - tr) + abs(nc - tc), ng, nr, nc))
            return -1

        total, sr, sc = 0, 0, 0
        for _, tr, tc in trees:
            d = astar(sr, sc, tr, tc)
            if d < 0:
                return -1
            total += d
            sr, sc = tr, tc
        return total

assert SolutionAStar().cutOffTree(ex1) == 6
assert SolutionAStar().cutOffTree(ex3) == 6
print("SolutionAStar OK")

SolutionAStar OK


**Độ phức tạp Cách 3:** xấu nhất (mê cung ngoằn ngoèo) như Dijkstra `O(mn log(mn))` mỗi cặp; trên lưới thoáng gần `O(độ dài đường đi)`. Space `O(mn)`.

### 7.3. Cách 4 — Thuật toán Hadlock (0-1 BFS theo số bước đi vòng)

**Giải thích.** Mỗi bước hoặc **đến gần** đích (Manhattan giảm 1) hoặc **đi xa** đích (Manhattan tăng 1). Nếu một đường đi có `D` bước đi xa thì số bước đến gần là `Manhattan + D`, tổng độ dài `= Manhattan + 2D`. Vậy **tối thiểu độ dài ⇔ tối thiểu `D`**.

Coi bước đến gần có chi phí 0, bước đi xa có chi phí 1 → bài toán đường ngắn nhất với trọng số 0/1 → **0-1 BFS**: chi phí 0 thì `appendleft`, chi phí 1 thì `append`.

In [14]:
def hadlock_dist(forest, sr, sc, tr, tc):
    if (sr, sc) == (tr, tc):
        return 0
    done = set()
    dq = deque([(0, sr, sc)])
    while dq:
        detours, r, c = dq.popleft()
        if (r, c) in done:
            continue
        done.add((r, c))
        if (r, c) == (tr, tc):
            return manhattan(sr, sc, tr, tc) + 2 * detours
        here = manhattan(r, c, tr, tc)
        for nr, nc in neighbors(forest, r, c):
            if (nr, nc) in done:
                continue
            if manhattan(nr, nc, tr, tc) < here:
                dq.appendleft((detours, nr, nc))       # bước đến gần: chi phí 0
            else:
                dq.append((detours + 1, nr, nc))       # bước đi xa: chi phí 1
    return -1

def cut_off_hadlock(forest):
    return total_steps_quiet(hadlock_dist, forest)

print(cut_off_hadlock(ex1), cut_off_hadlock(ex2), cut_off_hadlock(ex3), cut_off_hadlock(ex4))
print("Hadlock (0,0)->(2,0) trong ex1:", hadlock_dist(ex1, 0, 0, 2, 0), "= 2 + 2*2")
assert cut_off_hadlock(ex1) == 6 and cut_off_hadlock(ex2) == -1

6 -1 6 9
Hadlock (0,0)->(2,0) trong ex1: 6 = 2 + 2*2


**Đọc code:**

- `done`: ô đã "chốt" số bước đi vòng nhỏ nhất (giống Dijkstra). Một ô có thể nằm trong deque nhiều lần; chỉ lần lấy ra đầu tiên có giá trị.
- `appendleft` cho chi phí 0 giữ deque luôn sắp theo `detours` (chỉ có hai giá trị liền nhau `d` và `d + 1` trong deque).
- Kết quả `manhattan + 2 * detours`.

**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionHadlock:
    def cutOffTree(self, forest: List[List[int]]) -> int:
        m, n = len(forest), len(forest[0])
        if forest[0][0] == 0:
            return -1
        trees = sorted((v, r, c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1)

        def hadlock(sr, sc, tr, tc):
            if (sr, sc) == (tr, tc):
                return 0
            done = [[False] * n for _ in range(m)]
            dq = deque([(0, sr, sc)])
            while dq:
                d, r, c = dq.popleft()
                if done[r][c]:
                    continue
                done[r][c] = True
                if r == tr and c == tc:
                    return abs(sr - tr) + abs(sc - tc) + 2 * d
                here = abs(r - tr) + abs(c - tc)
                for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                    if 0 <= nr < m and 0 <= nc < n and forest[nr][nc] and not done[nr][nc]:
                        if abs(nr - tr) + abs(nc - tc) < here:
                            dq.appendleft((d, nr, nc))
                        else:
                            dq.append((d + 1, nr, nc))
            return -1

        total, sr, sc = 0, 0, 0
        for _, tr, tc in trees:
            d = hadlock(sr, sc, tr, tc)
            if d < 0:
                return -1
            total += d
            sr, sc = tr, tc
        return total

assert SolutionHadlock().cutOffTree(ex1) == 6
assert SolutionHadlock().cutOffTree(ex2) == -1
print("SolutionHadlock OK")

SolutionHadlock OK


**Độ phức tạp Cách 4:** mỗi ô được chốt một lần, mỗi ô đẩy tối đa 4 bản ghi → Time `O(k · m · n)`, Space `O(m · n)`. Trên lưới thoáng, Hadlock đi thẳng tới đích như A* nhưng không cần heap.

## 8. TRACE từng bước

### 8.1. Bảng các đoạn đường (ví dụ 4)

| Đoạn | Từ | Tới | Manhattan | BFS | số lần đi vòng (Hadlock) |
|---|---|---|---|---|---|
| start → 2 | (0,0) | (1,2) | 3 | 3 | 0 |
| 2 → 3 | (1,2) | (1,0) | 2 | 4 | 1 |
| 3 → 4 | (1,0) | (0,1) | 2 | 2 | 0 |

### 8.2. Trace BFS theo lớp

Hàm `trace_bfs` in các lớp BFS (tập ô ở khoảng cách `d`) và vẽ lưới khoảng cách khi tới đích.

In [16]:
def trace_bfs(forest, sr, sc, tr, tc):
    m, n = len(forest), len(forest[0])
    dist = [[None] * n for _ in range(m)]
    dist[sr][sc] = 0
    frontier, d = [(sr, sc)], 0
    print(f"BFS từ ({sr},{sc}) tới ({tr},{tc})")
    print(f"  lớp 0: {frontier}")
    found = (sr, sc) == (tr, tc)
    while frontier and not found:
        d += 1
        nxt = []
        for r, c in frontier:
            for nr, nc in neighbors(forest, r, c):
                if dist[nr][nc] is None:
                    dist[nr][nc] = d
                    nxt.append((nr, nc))
                    found |= (nr, nc) == (tr, tc)
        frontier = nxt
        print(f"  lớp {d}: {frontier}")
    for r in range(m):
        print("   " + ' '.join('##' if forest[r][c] == 0 else (' ?' if dist[r][c] is None else f"{dist[r][c]:>2}") for c in range(n)))
    return dist[tr][tc] if dist[tr][tc] is not None else -1

assert trace_bfs(ex4, 1, 2, 1, 0) == 4

BFS từ (1,2) tới (1,0)
  lớp 0: [(1, 2)]
  lớp 1: [(0, 2)]
  lớp 2: [(0, 1)]
  lớp 3: [(0, 0)]
  lớp 4: [(1, 0)]
    3  2  1
    4 ##  0


### 8.3. Trace toàn bộ quá trình chặt cây

In [17]:
def trace_cut(forest, dist_fn=bfs_dist):
    trees = sorted((v, r, c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1)
    sr = sc = total = 0
    print(f"{'cây':>4} | {'từ':>6} | {'tới':>6} | {'Manh.':>5} | {'dist':>4} | {'tổng':>4}")
    for v, tr, tc in trees:
        d = dist_fn(forest, sr, sc, tr, tc)
        if d < 0:
            print(f"{v:>4} | ({sr},{sc}) | ({tr},{tc}) | không tới được -> -1")
            return -1
        total += d
        print(f"{v:>4} | ({sr},{sc})  | ({tr},{tc})  | {manhattan(sr, sc, tr, tc):>5} | {d:>4} | {total:>4}")
        sr, sc = tr, tc
    return total

assert trace_cut(ex1) == 6
print()
assert trace_cut(ex2) == -1

 cây |     từ |    tới | Manh. | dist | tổng
   2 | (0,0)  | (0,1)  |     1 |    1 |    1
   3 | (0,1)  | (0,2)  |     1 |    1 |    2
   4 | (0,2)  | (1,2)  |     1 |    1 |    3
   5 | (1,2)  | (2,2)  |     1 |    1 |    4
   6 | (2,2)  | (2,1)  |     1 |    1 |    5
   7 | (2,1)  | (2,0)  |     1 |    1 |    6

 cây |     từ |    tới | Manh. | dist | tổng
   2 | (0,0)  | (0,1)  |     1 |    1 |    1
   3 | (0,1)  | (0,2)  |     1 |    1 |    2
   5 | (0,2) | (2,2) | không tới được -> -1


### 8.4. Trace Hadlock: deque và số lần đi vòng

In [18]:
def trace_hadlock(forest, sr, sc, tr, tc):
    done = set()
    dq = deque([(0, sr, sc)])
    step = 0
    while dq:
        detours, r, c = dq.popleft()
        if (r, c) in done:
            continue
        done.add((r, c))
        step += 1
        print(f"bước {step}: chốt ({r},{c}) với {detours} lần đi vòng | deque còn {list(dq)}")
        if (r, c) == (tr, tc):
            ans = manhattan(sr, sc, tr, tc) + 2 * detours
            print(f"tới đích: {manhattan(sr, sc, tr, tc)} + 2*{detours} = {ans}")
            return ans
        here = manhattan(r, c, tr, tc)
        for nr, nc in neighbors(forest, r, c):
            if (nr, nc) not in done:
                if manhattan(nr, nc, tr, tc) < here:
                    dq.appendleft((detours, nr, nc))
                else:
                    dq.append((detours + 1, nr, nc))
    return -1

assert trace_hadlock(ex4, 1, 2, 1, 0) == 4

bước 1: chốt (1,2) với 0 lần đi vòng | deque còn []
bước 2: chốt (0,2) với 1 lần đi vòng | deque còn []
bước 3: chốt (0,1) với 1 lần đi vòng | deque còn []
bước 4: chốt (0,0) với 1 lần đi vòng | deque còn []
bước 5: chốt (1,0) với 1 lần đi vòng | deque còn []
tới đích: 2 + 2*1 = 4


## 9. Code nộp LeetCode

Lời giải chính: **sắp cây + BFS theo lớp cho từng cặp** (Cách 2), có dấu thời gian để không cấp phát lại.

In [19]:
from collections import deque
from typing import List

class Solution:
    def cutOffTree(self, forest: List[List[int]]) -> int:
        m, n = len(forest), len(forest[0])
        if forest[0][0] == 0:
            return -1
        # Các cây theo chiều cao tăng dần, vị trí mã hóa r*n + c
        trees = sorted((v, r * n + c) for r, row in enumerate(forest)
                       for c, v in enumerate(row) if v > 1)
        walk = [forest[i // n][i % n] != 0 for i in range(m * n)]
        stamp = [0] * (m * n)
        cur_stamp = 0

        def bfs(src: int, dst: int) -> int:
            nonlocal cur_stamp
            if src == dst:
                return 0
            cur_stamp += 1
            stamp[src] = cur_stamp
            frontier, d = [src], 0
            while frontier:
                d += 1
                nxt = []
                for u in frontier:
                    r, c = divmod(u, n)
                    for v, ok in ((u - n, r > 0), (u + n, r < m - 1),
                                  (u - 1, c > 0), (u + 1, c < n - 1)):
                        if ok and walk[v] and stamp[v] != cur_stamp:
                            if v == dst:
                                return d
                            stamp[v] = cur_stamp
                            nxt.append(v)
                frontier = nxt
            return -1

        total, cur = 0, 0
        for _, pos in trees:
            d = bfs(cur, pos)
            if d < 0:
                return -1
            total += d
            cur = pos
        return total

sol = Solution()
assert sol.cutOffTree(ex1) == 6
assert sol.cutOffTree(ex2) == -1
assert sol.cutOffTree(ex3) == 6
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

1. **Thứ tự chặt duy nhất.** Chiều cao đôi một khác nhau, phải chặt theo thứ tự tăng → dãy `T1, ..., Tk` cố định.
2. **Khoảng cách không đổi theo thời gian.** Một ô đi được ⇔ giá trị khác 0. Chặt cây biến giá trị `> 1` thành `1` — vẫn khác 0. Vậy đồ thị các ô đi được **không đổi**, khoảng cách `dist(u, v)` là hằng số.
3. **Tách tổng.** Mọi hành trình hợp lệ ghé `T1, ..., Tk` theo thứ tự (có thể đi qua cây khác mà không chặt — được phép), nên chia thành `k` đoạn: đoạn thứ `i` đi từ `T(i-1)` (với `T0` = start) tới `Ti`, dài ít nhất `dist(T(i-1), Ti)`. Ngược lại, nối các đường ngắn nhất lại được một hành trình hợp lệ có độ dài đúng bằng tổng. Vậy đáp án = `Σ dist(T(i-1), Ti)`, hoặc `-1` nếu một khoảng cách vô cùng.
4. **BFS đúng** vì mọi cạnh dài 1: các ô được lấy ra theo khoảng cách tăng dần.
5. **A* đúng** vì heuristic Manhattan nhất quán (`h(u) <= 1 + h(v)` với `u, v` kề, và `h(đích) = 0`) → lần đầu lấy đích ra là tối ưu.
6. **Hadlock đúng** vì mọi đường đi từ `s` tới `t` có độ dài `Manhattan(s, t) + 2D` với `D` = số bước đi xa (mỗi bước làm Manhattan đổi đúng ±1); 0-1 BFS tìm `D` nhỏ nhất.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Floyd | `O((mn)^3)` | `O((mn)^2)` |
| 2. BFS | `O(mn log(mn) + k · mn)` | `O(mn)` |
| 3. A* | `O(k · mn log(mn))` xấu nhất | `O(mn)` |
| 4. Hadlock | `O(k · mn)` | `O(mn)` |

`k <= mn = 2500` → BFS cỡ `6·10^6` lượt thăm ô trong trường hợp xấu nhất. Cell dưới đo trên lưới `50 × 50` có 400 cây.

In [20]:
# Chọn lưới có vật cản nhưng vẫn chặt hết được (thử các seed tới khi đáp án != -1)
seed = 5
while True:
    random.seed(seed)
    big = rand_forest(50, 50, p_block=0.12, p_tree=0.16)
    big[0][0] = 1
    if Solution().cutOffTree(big) != -1:
        break
    seed += 1
print("seed:", seed)
print("số cây:", sum(v > 1 for row in big for v in row))
results = {}
for name, f in [("Solution(BFS)", Solution().cutOffTree), ("A*", SolutionAStar().cutOffTree), ("Hadlock", SolutionHadlock().cutOffTree)]:
    t = time.perf_counter()
    results[name] = f(big)
    print(f"{name:>14}: {results[name]}  {time.perf_counter() - t:.3f}s")
assert len(set(results.values())) == 1

seed:

 5
số cây: 373


 Solution(BFS): 12742  0.145s


            A*: 12742  0.091s
       Hadlock: 12742  0.021s


In [21]:
# Lưới thoáng (không vật cản): A* và Hadlock gần như đi thẳng
random.seed(6)
open_grid = rand_forest(50, 50, p_block=0.0, p_tree=0.16)
for name, f in [("Solution(BFS)", Solution().cutOffTree), ("A*", SolutionAStar().cutOffTree), ("Hadlock", SolutionHadlock().cutOffTree)]:
    t = time.perf_counter()
    r = f(open_grid)
    print(f"{name:>14}: {r}  {time.perf_counter() - t:.3f}s")

 Solution(BFS): 13345  0.186s


            A*: 13345  0.131s
       Hadlock: 13345  0.012s


## 12. Bộ test

Gồm: ví dụ đề, edge case (lưới 1×1, xuất phát là cây, cây bị cô lập, xuất phát trên vật cản, đường dích dắc) và đối chiếu ngẫu nhiên có seed giữa mọi cách với Floyd–Warshall.

In [22]:
def all_methods(forest):
    return {
        "floyd": cut_off_floyd(forest),
        "bfs": cut_off_bfs(forest),
        "astar": cut_off_astar(forest),
        "hadlock": cut_off_hadlock(forest),
        "SolutionBFS": SolutionBFS().cutOffTree(forest),
        "SolutionAStar": SolutionAStar().cutOffTree(forest),
        "SolutionHadlock": SolutionHadlock().cutOffTree(forest),
        "Solution": Solution().cutOffTree(forest),
    }

cases = [
    (ex1, 6), (ex2, -1), (ex3, 6), (ex4, 9),
    ([[5]], 0),                                   # 1x1, đứng sẵn ở cây
    ([[1, 2]], 1),
    ([[2, 1], [0, 3]], 2),                       # 2 tại (0,0), 3 tại (1,1): (0,0)->(0,1)->(1,1)
    ([[1, 0, 2]], -1),                            # cây bị chặn
    ([[1, 1, 1], [0, 0, 1], [3, 1, 1], [1, 0, 0], [1, 1, 2]], 14),   # đường dích dắc: 10 + 4
]
for forest, want in cases:
    for name, got in all_methods(forest).items():
        assert got == want, (forest, name, got, want)
print("Tất cả", len(cases), "test cố định OK")

Tất cả 9 test cố định OK


In [23]:
# Xuất phát trên vật cản: các bản nộp trả -1 (Floyd coi (0,0) là đỉnh cô lập nên cũng -1)
blocked_start = [[0, 2], [3, 1]]
print({k: v for k, v in all_methods(blocked_start).items() if not k.startswith(("bfs", "astar", "hadlock"))})
assert Solution().cutOffTree(blocked_start) == -1 == cut_off_floyd(blocked_start)

{'floyd': -1, 'SolutionBFS': -1, 'SolutionAStar': -1, 'SolutionHadlock': -1, 'Solution': -1}


In [24]:
random.seed(2024)
for trial in range(400):
    m, n = random.randint(1, 5), random.randint(1, 5)
    forest = rand_forest(m, n, p_block=random.choice([0.0, 0.2, 0.35]), p_tree=0.35)
    if forest[0][0] == 0:
        forest[0][0] = 1
    res = all_methods(forest)
    want = res["floyd"]
    for name, got in res.items():
        assert got == want, (forest, name, got, want)
print("400 lưới ngẫu nhiên: mọi cách khớp Floyd–Warshall")

400 lưới ngẫu nhiên: mọi cách khớp Floyd–Warshall


## 13. Lỗi người mới hay gặp

### Lỗi 1: Coi cây chưa chặt là vật cản

**Code sai:** chỉ cho đi qua ô `== 1`.

**Vì sao sai:** đề nói ô có cây **đi được**. Chặn cây làm khoảng cách dài ra hoặc thành `-1` sai.

In [25]:
def bfs_trees_block_wrong(forest, sr, sc, tr, tc):
    m, n = len(forest), len(forest[0])
    seen = {(sr, sc)}
    q = deque([(sr, sc, 0)])
    while q:
        r, c, d = q.popleft()
        if (r, c) == (tr, tc):
            return d
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen and (forest[nr][nc] == 1 or (nr, nc) == (tr, tc)):  # SAI
                seen.add((nr, nc))
                q.append((nr, nc, d + 1))
    return -1

g = [[1, 3, 2]]
print("sai :", total_steps_quiet(bfs_trees_block_wrong, g), "| đúng:", Solution().cutOffTree(g))

sai : -1 | đúng: 3


**Code đúng:** đi được ⇔ `forest[r][c] != 0`.

### Lỗi 2: Đánh dấu `visited` khi lấy ra thay vì khi thêm vào

**Code sai:** `seen.add` sau `popleft`. Kết quả vẫn đúng nhưng một ô có thể bị thêm vào hàng đợi **nhiều lần** → chậm gấp vài lần, có thể TLE.

In [26]:
def bfs_mark_late(forest, sr, sc, tr, tc, counter):
    seen = set()
    q = deque([(sr, sc, 0)])
    while q:
        r, c, d = q.popleft()
        if (r, c) in seen:
            continue
        seen.add((r, c))                 # đánh dấu muộn
        if (r, c) == (tr, tc):
            return d
        for nr, nc in neighbors(forest, r, c):
            if (nr, nc) not in seen:
                counter[0] += 1
                q.append((nr, nc, d + 1))
    return -1

def bfs_mark_early(forest, sr, sc, tr, tc, counter):
    seen = {(sr, sc)}
    q = deque([(sr, sc, 0)])
    while q:
        r, c, d = q.popleft()
        if (r, c) == (tr, tc):
            return d
        for nr, nc in neighbors(forest, r, c):
            if (nr, nc) not in seen:
                seen.add((nr, nc))       # đánh dấu sớm
                counter[0] += 1
                q.append((nr, nc, d + 1))
    return -1

grid = [[1] * 30 for _ in range(30)]
c1, c2 = [0], [0]
d1 = bfs_mark_late(grid, 0, 0, 29, 29, c1)
d2 = bfs_mark_early(grid, 0, 0, 29, 29, c2)
print(f"đánh dấu muộn: dist={d1}, số lần push={c1[0]} | đánh dấu sớm: dist={d2}, số lần push={c2[0]}")

đánh dấu muộn: dist=58, số lần push=1740 | đánh dấu sớm: dist=58, số lần push=899


**Code đúng:** đánh dấu ngay khi `append`.

### Lỗi 3: Quên trường hợp đứng sẵn ở cây (khoảng cách 0)

**Code sai:** BFS chỉ kiểm tra đích khi phát hiện ô **kề** → nếu `src == dst` thì không bao giờ khớp, trả `-1` hoặc một vòng đi về.

In [27]:
def bfs_no_zero_wrong(forest, sr, sc, tr, tc):
    m, n = len(forest), len(forest[0])
    seen = {(sr, sc)}
    q = deque([(sr, sc, 0)])
    while q:
        r, c, d = q.popleft()
        for nr, nc in neighbors(forest, r, c):
            if (nr, nc) == (tr, tc):
                return d + 1             # SAI khi src == dst
            if (nr, nc) not in seen:
                seen.add((nr, nc))
                q.append((nr, nc, d + 1))
    return -1

print("sai :", total_steps_quiet(bfs_no_zero_wrong, ex3), "| đúng:", Solution().cutOffTree(ex3))

sai : 8 | đúng: 6


**Code đúng:** `if src == dst: return 0` ở đầu hàm.

### Lỗi 4: Chặt theo thứ tự "gần nhất trước" (tham lam vị trí)

Đề bắt **chiều cao tăng dần**. Chọn cây gần nhất cho số bước nhỏ hơn nhưng **sai đề**. Với ví dụ 4, tham lam vị trí cho tổng nhỏ hơn 9 — nhưng đáp án đúng là 9.

## 14. Mẹo & nhận dạng pattern

- **"Lưới, mỗi bước 4 hướng, chi phí 1, tìm số bước ít nhất"** → BFS.
- **Nhiều điểm phải ghé theo thứ tự cố định** + đường đi không đổi theo thời gian → tổng các khoảng cách từng cặp.
- Thứ tự **không** cố định (tự chọn) → bitmask DP/TSP (ví dụ LC 847, 943, 1293...).
- Lưới thoáng, cần tăng tốc → A* với Manhattan, hoặc Hadlock (0-1 BFS).
- Cạnh trọng số 0/1 → 0-1 BFS bằng `deque` (`appendleft` / `append`).
- Mẹo cài đặt: mã hóa ô `r*n + c`, BFS theo lớp, mảng dấu thời gian.

## 15. Biến thể

1. **Thứ tự chặt tùy ý** (chặt hết với tổng bước ít nhất) → TSP trên `k` cây: BFS để có ma trận khoảng cách `k × k`, rồi bitmask DP `O(2^k · k^2)` — chỉ được khi `k` nhỏ.
2. **Cây là vật cản cho tới khi bị chặt** → khoảng cách thay đổi theo thời gian; vẫn tách theo cặp nhưng đồ thị đổi sau mỗi lần chặt.
3. **Đi 8 hướng** → thêm 4 hướng chéo; Manhattan không còn là heuristic hợp lệ cho A* (dùng Chebyshev).
4. **Trả về đường đi cụ thể** → BFS lưu `parent`, dựng ngược.

In [28]:
def cut_any_order(forest):
    """Biến thể 1: chặt hết theo thứ tự tùy ý, tổng bước ít nhất (bitmask DP)."""
    pts = [(0, 0)] + [(r, c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1]
    k = len(pts) - 1
    D = [[bfs_dist(forest, *pts[i], *pts[j]) for j in range(k + 1)] for i in range(k + 1)]
    INF = float('inf')
    dp = [[INF] * (k + 1) for _ in range(1 << k)]     # dp[mask][i]: đã chặt mask, đang ở cây i (1..k)
    for i in range(1, k + 1):
        if D[0][i] >= 0:
            dp[1 << (i - 1)][i] = D[0][i]
    for mask in range(1 << k):
        for i in range(1, k + 1):
            if dp[mask][i] == INF:
                continue
            for j in range(1, k + 1):
                if not mask >> (j - 1) & 1 and D[i][j] >= 0:
                    nm = mask | 1 << (j - 1)
                    dp[nm][j] = min(dp[nm][j], dp[mask][i] + D[i][j])
    best = min(dp[(1 << k) - 1][1:])
    return -1 if best == INF else best

print("ví dụ 4 — theo chiều cao:", Solution().cutOffTree(ex4), "| thứ tự tùy ý:", cut_any_order(ex4))
print("ví dụ 1 — theo chiều cao:", Solution().cutOffTree(ex1), "| thứ tự tùy ý:", cut_any_order(ex1))

ví dụ 4 — theo chiều cao: 9 | thứ tự tùy ý: 5
ví dụ 1 — theo chiều cao: 6 | thứ tự tùy ý: 6


In [29]:
def bfs_path(forest, sr, sc, tr, tc):
    """Biến thể 4: trả về danh sách ô trên một đường ngắn nhất."""
    parent = {(sr, sc): None}
    q = deque([(sr, sc)])
    while q:
        u = q.popleft()
        if u == (tr, tc):
            path = []
            while u is not None:
                path.append(u)
                u = parent[u]
            return path[::-1]
        for v in neighbors(forest, *u):
            if v not in parent:
                parent[v] = u
                q.append(v)
    return None

p = bfs_path(ex1, 0, 0, 2, 0)
print(p, "-> số bước", len(p) - 1)
assert len(p) - 1 == bfs_dist(ex1, 0, 0, 2, 0)

[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0)] -> số bước 6


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Floyd | khoảng cách mọi cặp ô | `O((mn)^3)` | `O((mn)^2)` | trọng tài cho lưới ≤ 6×6 |
| 2. BFS | BFS từng cặp cây liên tiếp | `O(k · mn)` | `O(mn)` | bản nộp chuẩn, dễ viết đúng |
| 3. A* | ưu tiên `g + Manhattan` | xấu nhất `O(k · mn log mn)` | `O(mn)` | lưới thoáng, đường dài |
| 4. Hadlock | 0-1 BFS theo số lần đi vòng | `O(k · mn)` | `O(mn)` | lưới thoáng, không muốn dùng heap |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `cutOffTree([[3, 1, 2]])`, `cutOffTree([[1, 2], [0, 0], [3, 1]])`, `cutOffTree([[4, 2, 3]])`?

**Bài 2 (tính tay).** Lập bảng trace (như mục 8.3) cho `[[1, 1, 4], [5, 0, 1], [2, 1, 3]]`.

**Bài 3 (điền TODO).** Hoàn thiện `grid_bfs(forest, src, dst)` (không nhìn code).

**Bài 4 (tự viết lại).** Viết lại toàn bộ lời giải `my_cut_off(forest)`.

**Bài 5 (mini challenge).** Viết `cut_with_path(forest)` trả về `(tổng bước, danh sách mọi ô đã đi qua theo thứ tự)`; kiểm tra số ô − 1 = tổng bước và các ô liên tiếp kề nhau.

In [30]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def grid_bfs(forest, src, dst):
    # TODO: if src == dst: return 0; deque; seen; 4 hướng; ô != 0
    pass

# Bài 4 — TODO
def my_cut_off(forest):
    # TODO: sắp cây theo chiều cao; cộng grid_bfs từng cặp; -1 nếu không tới được
    pass

# Bài 5 — TODO
def cut_with_path(forest):
    # TODO: dùng BFS có parent cho từng đoạn, nối đường đi (bỏ ô trùng ở chỗ nối)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: liệt kê cây theo chiều cao.
- Mức 2: `[[3,1,2]]`: start → 2 tại cột 2 (2 bước), → 3 tại cột 0 (2 bước).
- Mức 3: đáp án `4`, `-1` (hàng giữa chặn), `[[4,2,3]]`: 1 + 1 + 2 = `4`.

**Bài 2.**
- Mức 1: thứ tự cây 2 (2,0), 3 (2,2), 4 (0,2), 5 (1,0).
- Mức 2: ô `(1,1)` là vật cản, phải đi vòng.
- Mức 3: các đoạn: 2, 2, 2, 3 → tổng 9.

**Bài 3.**
- Mức 1: `seen = {src}`, `q = deque([(src, 0)])`.
- Mức 2: với mỗi ô kề hợp lệ chưa thấy: nếu là đích trả `d + 1`.
- Mức 3: hết hàng đợi trả `-1`.

**Bài 4.**
- Mức 1: `trees = sorted(...)`.
- Mức 2: `cur = (0, 0)`, `total = 0`.
- Mức 3: kiểm tra `forest[0][0] == 0` → `-1`.

**Bài 5.**
- Mức 1: viết `bfs_path` trả về list ô.
- Mức 2: nối `path[1:]` của mỗi đoạn sau đoạn đầu.
- Mức 3: kiểm tra `|r1-r2| + |c1-c2| == 1` cho mọi cặp liên tiếp.

## 19. Đáp án tham khảo

In [31]:
# Bài 1
assert Solution().cutOffTree([[3, 1, 2]]) == 4
assert Solution().cutOffTree([[1, 2], [0, 0], [3, 1]]) == -1
assert Solution().cutOffTree([[4, 2, 3]]) == 4

# Bài 2
g2 = [[1, 1, 4], [5, 0, 1], [2, 1, 3]]
assert trace_cut(g2) == 9

# Bài 3
def grid_bfs_ans(forest, src, dst):
    if src == dst:
        return 0
    m, n = len(forest), len(forest[0])
    seen = {src}
    q = deque([(src, 0)])
    while q:
        (r, c), d = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and forest[nr][nc] and (nr, nc) not in seen:
                if (nr, nc) == dst:
                    return d + 1
                seen.add((nr, nc))
                q.append(((nr, nc), d + 1))
    return -1

assert grid_bfs_ans(ex1, (0, 0), (2, 0)) == 6
print("Bài 1–3 OK")

 cây |     từ |    tới | Manh. | dist | tổng
   2 | (0,0)  | (2,0)  |     2 |    2 |    2
   3 | (2,0)  | (2,2)  |     2 |    2 |    4
   4 | (2,2)  | (0,2)  |     2 |    2 |    6
   5 | (0,2)  | (1,0)  |     3 |    3 |    9
Bài 1–3 OK


In [32]:
# Bài 4
def my_cut_off_ans(forest):
    if forest[0][0] == 0:
        return -1
    trees = sorted((v, r, c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1)
    cur, total = (0, 0), 0
    for _, r, c in trees:
        d = grid_bfs_ans(forest, cur, (r, c))
        if d < 0:
            return -1
        total += d
        cur = (r, c)
    return total

# Bài 5
def cut_with_path_ans(forest):
    trees = sorted((v, r, c) for r, row in enumerate(forest) for c, v in enumerate(row) if v > 1)
    cur, full = (0, 0), [(0, 0)]
    for _, r, c in trees:
        seg = bfs_path(forest, *cur, r, c)
        if seg is None:
            return -1, None
        full.extend(seg[1:])
        cur = (r, c)
    return len(full) - 1, full

steps, path = cut_with_path_ans(ex1)
print(steps, path)
assert steps == 6
random.seed(10)
for _ in range(300):
    g = rand_forest(random.randint(1, 5), random.randint(1, 5))
    if g[0][0] == 0:
        g[0][0] = 1
    assert my_cut_off_ans(g) == cut_off_floyd(g)
    s, p = cut_with_path_ans(g)
    assert s == cut_off_floyd(g)
    if p:
        assert all(abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1 for a, b in zip(p, p[1:]))
print("Bài 4–5 OK (đối chiếu Floyd–Warshall)")

6 [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0)]
Bài 4–5 OK (đối chiếu Floyd–Warshall)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "ghé các điểm theo thứ tự cố định trên lưới":**

```python
trees = sorted((v, r, c) for r, row in enumerate(g) for c, v in enumerate(row) if v > 1)
cur, total = (0, 0), 0
for _, r, c in trees:
    d = bfs(cur, (r, c))       # 0 nếu cur == đích, -1 nếu không tới được
    if d < 0: return -1
    total += d; cur = (r, c)
return total
```

**Template BFS lưới:**

```python
seen = {src}; q = deque([(src, 0)])
while q:
    (r, c), d = q.popleft()
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n and g[nr][nc] and (nr, nc) not in seen:
            if (nr, nc) == dst: return d + 1
            seen.add((nr, nc)); q.append(((nr, nc), d + 1))
return -1
```

**Checklist:**

- [ ] Ô có cây **đi được** (`!= 0`), không phải chỉ `== 1`.
- [ ] Chặt theo **chiều cao tăng dần**, không theo vị trí.
- [ ] `src == dst` → 0 bước (ô xuất phát có thể là cây).
- [ ] Đánh dấu `visited` khi **thêm** vào hàng đợi.
- [ ] Trả `-1` ngay khi một đoạn không tới được.
- [ ] Kiểm tra biên `0 <= nr < m`, `0 <= nc < n`.